# RawGrad — Optimizer Comparison

This notebook compares optimization algorithms implemented in the RawGrad framework.

We will compare:

- Stochastic Gradient Descent (SGD)
- Momentum

Both optimizers will train the same neural-network architecture on the same dataset.

The experiment measures:

- Training loss
- Training accuracy
- Test loss
- Test accuracy

The goal is to understand how the choice of optimizer affects neural-network training.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from rawgrad.layers import Dense
from rawgrad.activations import ReLU, Softmax
from rawgrad.model import Sequential
from rawgrad.classification_losses import CategoricalCrossEntropy
from rawgrad.optimizers import SGD, Momentum

In [2]:
np.random.seed(42)

X_train = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 1.0, 0.5],
    [0.0, 1.0, 2.0],
    [3.0, 2.0, 1.0],
    [1.5, 2.5, 3.0],
    [2.5, 1.0, 0.5],
    [0.2, 1.2, 2.2],
    [3.2, 2.0, 1.0],
])

y_train = np.array([0, 1, 2, 1, 0, 1, 2, 1])

X_test = np.array([
    [1.2, 2.1, 2.8],
    [2.2, 1.1, 0.4],
    [0.1, 1.1, 2.1],
    [3.1, 1.9, 1.1],
])

y_test = np.array([0, 1, 2, 1])

num_classes = 3

Y_train = np.zeros((len(y_train), num_classes))
Y_train[np.arange(len(y_train)), y_train] = 1.0

Y_test = np.zeros((len(y_test), num_classes))
Y_test[np.arange(len(y_test)), y_test] = 1.0

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))
print("Input features  :", X_train.shape[1])
print("Classes         :", num_classes)

Training samples: 8
Testing samples : 4
Input features  : 3
Classes         : 3


In [3]:
def create_model():
    return Sequential([
        Dense(3, 8, initialization="xavier"),
        ReLU(),
        Dense(8, 3, initialization="xavier"),
        Softmax(),
    ])

In [4]:
def train_model(model, optimizer, epochs=200):
    loss_function = CategoricalCrossEntropy()

    loss_history = []
    accuracy_history = []

    for epoch in range(epochs):
        predictions = model.forward(X_train)

        loss = loss_function.forward(
            predictions,
            Y_train,
        )

        gradient = loss_function.backward()

        model.backward(gradient)

        optimizer.step(model)

        predicted_classes = np.argmax(
            predictions,
            axis=1,
        )

        accuracy = np.mean(
            predicted_classes == y_train
        )

        loss_history.append(loss)
        accuracy_history.append(accuracy)

    return loss_history, accuracy_history

In [7]:
np.random.seed(42)

sgd_model = create_model()

sgd_optimizer = SGD(
    learning_rate=0.1
)

sgd_loss, sgd_accuracy = train_model(
    sgd_model,
    sgd_optimizer,
    epochs=200,
)

print("SGD training completed.")
print("Initial loss:", sgd_loss[0])
print("Final loss:  ", sgd_loss[-1])
print("Final accuracy:", f"{sgd_accuracy[-1] * 100:.2f}%")

SGD training completed.
Initial loss: 0.7890724264340684
Final loss:   0.09579587950627516
Final accuracy: 100.00%


In [8]:
np.random.seed(42)

momentum_model = create_model()

momentum_optimizer = Momentum(
    learning_rate=0.1
)

momentum_loss, momentum_accuracy = train_model(
    momentum_model,
    momentum_optimizer,
    epochs=200,
)

print("Momentum training completed.")
print("Initial loss:", momentum_loss[0])
print("Final loss:  ", momentum_loss[-1])
print("Final accuracy:", f"{momentum_accuracy[-1] * 100:.2f}%")

Momentum training completed.
Initial loss: 0.7890724264340684
Final loss:   0.0014481337619259372
Final accuracy: 100.00%


In [9]:
np.random.seed(42)

sgd_model = create_model()

sgd_optimizer = SGD(
    learning_rate=0.1
)

sgd_loss, sgd_accuracy = train_model(
    sgd_model,
    sgd_optimizer,
    epochs=200,
)

print("SGD training completed.")
print("Initial loss:", sgd_loss[0])
print("Final loss:  ", sgd_loss[-1])
print("Final accuracy:", f"{sgd_accuracy[-1] * 100:.2f}%")

SGD training completed.
Initial loss: 0.7890724264340684
Final loss:   0.09579587950627516
Final accuracy: 100.00%


In [10]:
print("OPTIMIZER COMPARISON")
print("-" * 40)

print(
    f"SGD      | "
    f"Loss: {sgd_loss[-1]:.6f} | "
    f"Accuracy: {sgd_accuracy[-1] * 100:.2f}%"
)

print(
    f"Momentum | "
    f"Loss: {momentum_loss[-1]:.6f} | "
    f"Accuracy: {momentum_accuracy[-1] * 100:.2f}%"
)

OPTIMIZER COMPARISON
----------------------------------------
SGD      | Loss: 0.095796 | Accuracy: 100.00%
Momentum | Loss: 0.001448 | Accuracy: 100.00%


In [12]:
loss_function = CategoricalCrossEntropy()

sgd_test_predictions = sgd_model.forward(X_test)
sgd_test_loss = loss_function.forward(
    sgd_test_predictions,
    Y_test,
)

sgd_test_classes = np.argmax(
    sgd_test_predictions,
    axis=1,
)

sgd_test_accuracy = np.mean(
    sgd_test_classes == y_test
)

momentum_test_predictions = momentum_model.forward(X_test)
momentum_test_loss = loss_function.forward(
    momentum_test_predictions,
    Y_test,
)

momentum_test_classes = np.argmax(
    momentum_test_predictions,
    axis=1,
)

momentum_test_accuracy = np.mean(
    momentum_test_classes == y_test
)

print("TEST EVALUATION")
print("-" * 40)
print(
    f"SGD      | "
    f"Loss: {sgd_test_loss:.6f} | "
    f"Accuracy: {sgd_test_accuracy * 100:.2f}%"
)
print(
    f"Momentum | "
    f"Loss: {momentum_test_loss:.6f} | "
    f"Accuracy: {momentum_test_accuracy * 100:.2f}%"
)

TEST EVALUATION
----------------------------------------
SGD      | Loss: 0.089360 | Accuracy: 100.00%
Momentum | Loss: 0.000642 | Accuracy: 100.00%


In [13]:
assert np.all(np.isfinite(sgd_loss))
assert np.all(np.isfinite(momentum_loss))

assert sgd_loss[-1] < sgd_loss[0]
assert momentum_loss[-1] < momentum_loss[0]

assert sgd_test_predictions.shape == (4, 3)
assert momentum_test_predictions.shape == (4, 3)

assert np.all(np.isfinite(sgd_test_predictions))
assert np.all(np.isfinite(momentum_test_predictions))

assert np.allclose(
    np.sum(sgd_test_predictions, axis=1),
    1.0,
)

assert np.allclose(
    np.sum(momentum_test_predictions, axis=1),
    1.0,
)

print("OPTIMIZERS: PASS")

OPTIMIZERS: PASS
